Make color cutouts from DP2 deep coadd images.

Based on DP2 tutorials 103.5, 103.6, 202.1, Padma's notebook. The cutout has a PSF model.

Please run it on the [RSP](https://data.lsst.cloud). 

In [ ]:
import lsst.afw.display as afw_display
from lsst.images.serialization import read_archive
from lsst.rsp import RSPDiscovery
from lsst.rsp.utils import get_pyvo_auth

from pyvo.dal.adhoc import SodaQuery

import io
import matplotlib.pyplot as plt
import numpy as np
from astropy.visualization import AsinhStretch, ImageNormalize, make_lupton_rgb
from astropy import units as u

%matplotlib inline

In [ ]:
def get_dl_result(band, ra, dec, radius):

    circle = (ra, dec, radius)

    results = sia_client.search(pos=circle, calib_level=3,
                                dpsubtype='lsst.deep_coadd')
    print("Num of sia_client search result: ", len(results))
    #t = results.to_table()
    #t.pprint_all()

    band_arr = results['lsst_band']
    #print(f'band_arr: {band_arr}')

    try:
        index = int(np.where(band_arr == band)[0][0])
    except:
        print(f'Band {band} does not exist!')
        return None

    dl_result = discovery.get_datalink_results(results[index])
    
    print(f"Datalink status: {dl_result.status}.")

    return dl_result
    

def get_cutout(band, ra, dec, radius=0.0015):

    dl_result = get_dl_result(band, ra, dec, radius)

    if dl_result is None:
        return None

    sq = SodaQuery.from_resource(dl_result,
                                 dl_result.get_adhocservice_by_id("cutout-sync-exposure"),
                                 session=get_pyvo_auth())

    sq.circle = (ra * u.deg, dec * u.deg, radius * u.deg)

    cutout_bytes = sq.execute_stream().read()
    try:
        sq.raise_if_error()
    except:
        print(f'Error in get cutout at band {band}!')
        return None

    cutout = read_archive(io.BytesIO(cutout_bytes))
    print('Cutout size [pix]: ', np.shape(cutout.image.array))

    return cutout


def save_cutout(filename, cutout):

    half_size = np.shape(cutout_tmp.image.array)[0] // 2
    psf_kernel_image = cutout.psf.compute_kernel_image(x=cutout.yx0.x+half_size,y=cutout.yx0.y+half_size)

    np.savez_compressed(
        filename,
        image=cutout.image.array,
        psf=psf_kernel_image.array,
    )

    # Or save the image and the psf separately
    # tag = filename.split('/')[-1].split('.')[-2]
    # np.save(f'observations/{tag}.npy', cutout.image.array)
    # np.save(f'psfs/{tag}_psf.npy', psf_kernel_image.array)
    #print(cutout.image.array, psf_kernel_image.array)
    print(f'Saved cutout to {filename}!')
    
    return 0

In [ ]:
def normalize_band(image, asinh_a=0.002):
    
    data = image.image.array

    vmin = -0.03
    #vmax = 200
    vmax = 3000

    norm = ImageNormalize(vmin=vmin, vmax=vmax,
                          stretch=AsinhStretch(a=asinh_a),
                          clip=True,
                         )

    scaled = norm(data)
    return scaled


def combine_RGB(R_image, G_image, B_image):
    
    R_channel = normalize_band(R_image)
    G_channel = normalize_band(G_image)
    B_channel = normalize_band(B_image)

    RGB_image = np.dstack([R_channel, G_channel, B_channel])

    return RGB_image

    

In [ ]:
def plot_afw(image):

    fig, ax = plt.subplots(figsize=(6,6))
    display = afw_display.Display(frame=fig)
    #display.scale('asinh', 'zscale')
    #display.scale('linear', 'zscale')
    display.scale('linear', -75, 125)
    display.image(image)

    return 0

In [ ]:
def plot_RGB(R_image, G_image, B_image, make_lupton=False, lim=None):

    if R_image is None or G_image is None or B_image is None:
        print('Missing data!')
        return 1

    RGB_image = combine_RGB(R_image, G_image, B_image)
    if make_lupton:
        RGB_image = make_lupton_rgb(R_image.image.array,
                                    G_image.image.array,
                                    B_image.image.array,
                                    stretch=0.002, Q=0.001)

    if lim is not None:
        xmin, xmax, ymin, ymax = lim
        RGB_image = RGB_image[xmin:xmax, ymin:ymax]
    
    fig = plt.figure(figsize=(6,6))
    im = plt.imshow(RGB_image,
                    origin='lower')

    return 0

In [ ]:
afw_display.setDefaultBackend("matplotlib")

discovery = RSPDiscovery("dp2")
sia_client = discovery.get_sia_client()

In [ ]:
# DESJ0407-5006
#target_ra, target_dec = 61.792580, -50.100250

# CXCOJ100201.50+020330.0
#target_ra, target_dec = 150.506300, 2.058100

# J0011-0845
target_ra, target_dec = 2.834350, -8.764070

In [ ]:
cutout_u = get_cutout('u', target_ra, target_dec)
cutout_g = get_cutout('g', target_ra, target_dec)
cutout_r = get_cutout('r', target_ra, target_dec)
cutout_i = get_cutout('i', target_ra, target_dec)
cutout_z = get_cutout('z', target_ra, target_dec)
cutout_y = get_cutout('y', target_ra, target_dec)

In [ ]:
plot_afw(cutout_g)

In [ ]:
cutout_tmp = cutout_g
half_size = np.shape(cutout_tmp.image.array)[0] // 2
psf_kernel_image = cutout_tmp.psf.compute_kernel_image(x=cutout_tmp.yx0.x+half_size,y=cutout_tmp.yx0.y+half_size)
psf_stellar_image = cutout_tmp.psf.compute_stellar_image(x=cutout_tmp.yx0.x+half_size,y=cutout_tmp.yx0.y+half_size)

In [ ]:
np.shape(cutout_tmp.image.array)

In [ ]:
np.shape(psf_kernel_image .array)

In [ ]:
save_cutout('cutout_g.npz', cutout_g)
save_cutout('cutout_r.npz', cutout_r)
save_cutout('cutout_i.npz', cutout_i)

In [ ]:
#tmp = np.load('cutout_g.npz')
#tmp = np.load('cutout_r.npz')
#tmp = np.load('cutout_i.npz')
#print(tmp['image'], tmp['psf'])

In [ ]:
plt.figure()
plt.imshow(psf_kernel_image.array)
#plt.imshow(psf_stellar_image.array)
plt.colorbar()
#display = afw_display.Display(frame=2)
#display.scale('linear', -0.01, 0.03)
#display.image(psf_kernel_image)

In [ ]:
#cutout_g.yx0
#cutout_g.bbox
#cutout_g.bounds

In [ ]:
plot_RGB(cutout_r, cutout_g, cutout_u)
plot_RGB(cutout_i, cutout_r, cutout_g)
plot_RGB(cutout_z, cutout_i, cutout_r)
plot_RGB(cutout_y, cutout_z, cutout_i)

In [ ]:
plot_RGB(cutout_r, cutout_g, cutout_u, True)
plot_RGB(cutout_i, cutout_r, cutout_g, True)
plot_RGB(cutout_z, cutout_i, cutout_r, True)
plot_RGB(cutout_y, cutout_z, cutout_i, True)

In [ ]:
plot_RGB(cutout_y, cutout_r, cutout_g, True)
plot_RGB(cutout_z, cutout_g, cutout_u, True)